# M13 lab: build a nation's pathway

*Energy Systems, Step by Step* · Modeler tier · lab for lesson M13 · about 60 minutes, plus 120 minutes of practice at the end.

You will rebuild the lesson's engine in Python and solve it with the open-source HiGHS solver through `scipy.optimize.linprog`:

1. Prairieland: five regions, the 2025 fleet with its retirement years, seven links, twelve time slices.
2. The plan: demand, electrification, the carbon budget, the import cap, costs (the browser's defaults).
3. The pathway as one linear program over 2030–2050: vintages, residual value, links, batteries, the stress hours, fuels, CO₂, the budget and the cap.
4. Solve the central plan and check it against the browser.
5. The budget's dual, and its price in each model year.
6. Deeper cuts, and a budget out of reach.
7. Without the peak-hour rule.
8. Your pathway (capstone C4).

**How to run it.**

1. Download this notebook and `prairieland.json`, `e4-pathways.json` and `e4-library.json` from the lesson page; put them in one folder.
2. In Google Colab: *File → Upload notebook*, then upload the three data files with the folder icon on the left.
3. Or locally: Python 3.9 or later with `numpy` and `scipy` 1.9 or later.
4. *Runtime → Run all* (or run every cell). The main part takes under a minute; the practice cells solve a few more linear programs.

**The data.** Prairieland is fictional; its numbers are illustrative. New plants' costs are EIA's 2026 assumptions (2025 dollars), fuel prices EIA's 2025 averages, CO₂ factors the IPCC's 2006 defaults, hourly shapes SPP's 2025 grid. The fleet, the regions, the links, the existing plants' operating costs, the penalties and the peak-hour rule are set for teaching.

**What the asserts check.** The cost to 2050, the cumulative CO₂, CO₂ and imports in each model year, what is built, and the budget's dual, against the numbers the page shows (`e4-library.json`, made with the page's engine).

In [ ]:
import copy
import json
import math
from pathlib import Path

import numpy as np
import scipy
from scipy.optimize import linprog
from scipy.sparse import coo_matrix

major, minor = (int(x) for x in scipy.__version__.split(".")[:2])
assert (major, minor) >= (1, 9), "this lab needs SciPy 1.9 or later (pip install -U scipy)"
for f in ("prairieland.json", "e4-pathways.json", "e4-library.json"):
    assert Path(f).exists(), f + " not found: put it next to this notebook (in Colab, upload it with the folder icon)."
PL = json.load(open("prairieland.json", encoding="utf-8"))
CFG = json.load(open("e4-pathways.json", encoding="utf-8"))
LIB = json.load(open("e4-library.json", encoding="utf-8"))
print(PL["about"]["line"])
print("NumPy", np.__version__, "· SciPy", scipy.__version__, "· library made on", LIB["date"])

## 1. Prairieland, prepared

- **Regions.** Northern Highlands, Ridge, Western Plains, Capital Valley, South Coast; each with its 2025 final energy by sector and carrier, its wind and solar capacity factors and the most wind and solar it can hold.
- **Model years.** 2030, 2035, 2040, 2045, 2050; each stands for the five years that end in it, so its yearly costs count five times, each year discounted to 2025.
- **Slices.** Twelve: four seasons × day and night, the peak net-load day and the scarce-wind winter week. The two extreme periods get slices of their own: each its average day and night, repeated over its days, so their hourly sequence is not kept (the page's full-year check runs every hour in order). `h` is the hours a slice stands for (they add to 8,760); `demand` scales each region's average load; `wind` and `solar` are output per MW.
- **Representative days.** A season's day slice and the night slice after it form one day that stands for `days` days; batteries cycle within it.
- **The fleet.** Coal, gas and oil plants unit by unit (each with its efficiency and retirement year); hydro, wind, solar, batteries and the existing nuclear plants by region. A plant counts in a model year only if its retirement year is later, so it leaves in the five-year period that contains that year: a conservative simplification (up to five years less of each retiring plant).
- **Links.** Seven, each with a transfer limit either way and losses of 1% per 100 km (a transport model: no power-flow physics).

In [ ]:
REG = ["highlands", "ridge", "plains", "capital", "coast"]
NEW = ["wind", "solar", "battery", "gasCC", "gasCT"]          # built region by region; "firm" is national
FOSSIL = ["coal", "gasCC", "gasCT", "oil"]
SEC = ["industry", "buildings", "transport", "agriculture"]
PJ_PER_MWH, MWH_PER_PJ = 3.6e-6, 1e6 / 3.6

def crf(r, n):
    return 1 / n if r == 0 else r / (1 - (1 + r) ** -n)

Y = PL["periods"]["years"]; NP = len(Y); EACH = PL["periods"]["yearsEach"]; BASE = PL["periods"]["base"]
R = []
for i, r in enumerate(PL["regions"]):
    R.append(dict(key=r["key"], name=r["name"], i=i, res=r["resources"], final=r["final"], el2025=sum(r["final"][k].get("el", 0) for k in SEC)))
RI = {r["key"]: r["i"] for r in R}
EL0 = sum(r["el2025"] for r in R)
for r in R:
    r["elShare"] = r["el2025"] / EL0
SL = []
for i, s in enumerate(PL["timeSlices"]["slices"]):
    SL.append(dict(key=s["key"], i=i, h=s["hours"], days=s["days"], part=s["part"], season=s["season"], demand=s["demand"],
                   wind=[s["wind"][k] for k in REG], solar=[s["solar"][k] for k in REG], hydro=s["hydro"], summer=s["season"] == "summer"))
DAYS = []
for s in SL:
    if s["part"] != "day": continue
    stem = s["key"][:-4]; n = next(x for x in SL if x["key"] == stem + "_night")
    DAYS.append(dict(key=stem, d=s["i"], n=n["i"], days=s["days"], week=stem == "lowwind"))
WEEK = next(d for d in DAYS if d["week"])
# the representative day of the week's season (winter) pays back what the scarce-wind week's store draws
REFILL = next(d for d in DAYS if not d["week"] and d["key"] != "peak" and SL[d["d"]]["season"] == SL[WEEK["d"]]["season"])
HOURS = sum(s["h"] for s in SL)
assert HOURS == 8760 and len(DAYS) == 6
FLEET = PL["fleet"]["units"]; AV = CFG["existing"]["availability"]; OM = CFG["existing"]["om"]
UNITS = [dict(id=u["id"], r=RI[u["region"]], tech=u["tech"], mw=u["mw"], retire=u["retire"], eff=u["efficiency"], av=AV[u["tech"]])
         for u in FLEET if u["tech"] in FOSSIL]
for j, u in enumerate(UNITS): u["j"] = j

def on(u, y): return u["retire"] > y
def exist(tech, r, y): return sum(u["mw"] for u in FLEET if u["tech"] == tech and RI[u["region"]] == r and u["retire"] > y)

LINKS = [dict(key=l["key"], a=RI[l["from"]], b=RI[l["to"]], mw=l["mw"], km=l["km"], loss=l["lossShare"], i=i) for i, l in enumerate(PL["transmission"]["links"])]
TX = PL["transmission"]["expansion"]
T = PL["costs"]["technologies"]; FUELS = PL["costs"]["fuels"]; LIM = PL["costs"]["buildLimits"]
FAC = PL["carbon"]["factors"]["tPerTJ"]                     # t per TJ = Mt per 1,000 PJ
TR = PL["balance"]["transformation"]
REF_YIELD = TR["refinery"]["out"] / TR["refinery"]["in"]
GAS_OWN = TR["ownUse"]["gas"] / PL["balance"]["primary"]["gas"]["domestic"]
COAL_OWN = TR["ownUse"]["coal"] / PL["balance"]["primary"]["coal"]
GRID_LOSS = TR["grid"]["lossShare"]
DOM_GAS = PL["balance"]["primary"]["gas"]["domestic"]; DOM_CRUDE = PL["balance"]["primary"]["oil"]["domestic"]; COAL_MAX = PL["balance"]["primary"]["coal"]
USEFUL = PL["balance"]["useful"]["efficiency"]; ELEC = PL["demand"]["electrification"]
HYDRO_CF = [r["res"].get("hydroCF", 0) for r in R]
SHOCKS = {s["key"]: s for s in PL["shocks"]}
FB = PL["balance"]["final"]
SIGMA0 = {k: FB[k].get("el", 0) * USEFUL[k]["el"] / sum(FB[k][c] * USEFUL[k][c] for c in FB[k]) for k in SEC}
for y in Y:   # the fleet online in each model year is prairieland.json's capacityByYearMW
    for t in ["coal", "gasCC", "gasCT", "oil", "nuclear", "hydro", "wind", "solar", "battery"]:
        assert sum(exist(t, r, y) for r in range(5)) == PL["fleet"]["capacityByYearMW"][str(y)][t]
print("electricity's 2025 share of useful energy:", {k: f"{v:.1%}" for k, v in SIGMA0.items()})

## 2. The plan, and the system it describes

The plan holds the levers of the Pathway Explorer (SB4). `DEFAULT` is the page's default plan (the "central" preset). `sys_of(plan)` turns it into what the linear program needs:

- **Discounting.** `D[p]` adds the discount factors of the five years model year `p` stands for: 2030 stands for 2026–2030.
- **Budget weights.** CO₂ of 2026–2050 = 2 × 2025's + 5, 5, 5, 5, 3 × each model year's (straight lines between model years: prairieland.json's linear weights).
- **Electrification.** A sector's share of fossil useful energy switched by 2050 (`z`) is set so that electricity's national share reaches the plan's; it rises in a straight line from 2025. Each GJ of fuel switched delivers its useful energy from electricity at the sector's ratio (heat pumps 3, EVs 0.89, industry 1.5).
- **Demand.** Useful energy grows by the scenario's % a year; new large loads (factories and data centers, loads only) are spread over the regions by their 2025 electricity.
- **Loads.** A region's average busbar load (MW) is its yearly electricity divided by 0.94 (grid losses are 6% of what enters the grid), over 8,760 hours; each slice scales it by `demand`.

In [ ]:
L = CFG["levers"]; B = PL["carbon"]["budget"]
DEFAULT = dict(dem=PL["demand"]["default"], eb=L["electrification"]["buildings"]["def"], et=L["electrification"]["transport"]["def"],
               ei=L["electrification"]["industry"]["def"], bud=L["budget"]["def"], acc=PL["carbon"]["accounting"]["default"], cap=L["importCap"]["def"],
               fs="perfect", rm=L["reserve"]["def"], fm=1, tx=1, rt=1, sh="none",
               cw=T["wind"]["capex"]["def"], cs=T["solar"]["capex"]["def"], cb=T["battery"]["capex"]["def"], cc=T["gasCC"]["capex"]["def"],
               ct=T["gasCT"]["capex"]["def"], cf=T["firm"]["capex"]["def"], cx=TX["costPerMWkm"]["def"],
               pg=FUELS["gas"]["def"], pl=FUELS["lng"]["def"], pc=FUELS["coal"]["def"], po=FUELS["oil"]["def"], dr=PL["periods"]["discount"]["def"])
PRESET = {p["id"]: dict(DEFAULT, **p["state"]) for p in CFG["presets"]}
assert PRESET["central"] == DEFAULT, "the page's default plan is the central preset"

def dsum(rate, y):
    return sum((1 + rate) ** -(t - 2025) for t in range(y - EACH + 1, y + 1))

def new_loads(NLd, y):
    ks = sorted(int(k) for k in NLd)
    for a, b in zip(ks, ks[1:]):
        if a <= y <= b: return NLd[str(a)] + (NLd[str(b)] - NLd[str(a)]) * (y - a) / (b - a)
    return 0.0 if y < ks[0] else NLd[str(ks[-1])]

def sys_of(s, shock=None, only=None, fixed=None, budget_left=None, horizon_end=None, fixed_through=-1):
    rate = s["dr"] / 100
    scen = next(x for x in PL["demand"]["scenarios"] if x["key"] == s["dem"])
    acc = PL["carbon"]["accounting"]; w = acc["stepWeights"] if s["acc"] == "step" else acc["linearWeights"]
    S = dict(state=dict(s), rate=rate, D=[dsum(rate, y) for y in Y], w=[w[str(y)] for y in Y], w0=w["2025"], budget=s["bud"], cap=s["cap"] / 100,
             firm=bool(s["fm"]), txOn=bool(s["tx"]), retOn=bool(s["rt"]), reserve=None if s["rm"] == "off" else {"r0": 0, "r10": 0.1, "r20": 0.2}[s["rm"]],
             capex=[dict(wind=s["cw"], solar=s["cs"], battery=s["cb"], gasCC=s["cc"], gasCT=s["ct"], firm=s["cf"]) for _ in Y],
             price=[dict(gas=s["pg"], lng=s["pl"], coal=s["pc"], oil=s["po"]) for _ in Y], txCost=s["cx"],
             hydroX=[1.0] * NP, summerX=[1.0] * NP, elX=[1.0] * NP, shock=shock)
    if shock:
        q = SHOCKS[shock]["params"]; pi = Y.index(CFG["levers"]["shockYear"][shock]); S["shockP"] = pi
        if shock == "gasSpike": S["price"][pi]["lng"] *= q["lngX"]; S["price"][pi]["gas"] *= q["gasX"]
        if shock == "drought": S["hydroX"][pi] = q["hydroX"]; S["summerX"][pi] = q["thermalSummerX"]
        if shock == "mineralBan": S["capex"][pi]["battery"] *= q["batteryCapexX"]
        if shock == "demandSurprise":
            for p, y in enumerate(Y):
                if y >= q["from"]: S["elX"][p] = q["elX"]
    zOf = lambda k, sig: max(0.0, (sig - SIGMA0[k]) / (1 - SIGMA0[k]))
    Z = dict(buildings=zOf("buildings", s["eb"] / 100), transport=zOf("transport", s["et"] / 100), industry=zOf("industry", s["ei"] / 100), agriculture=0.0)
    S["z"], S["elPJ"], S["fuel"], S["loadMW"], S["share"] = Z, [], [], [], []
    for p, y in enumerate(Y):
        fuel, el = dict(oil=0.0, gas=0.0, coal=0.0), []
        for r in R:
            e = 0.0
            for k in SEC:
                g = (1 + scen["growth"][k] / 100) ** (y - BASE); z = Z[k] * (y - BASE) / (2050 - BASE); F = r["final"][k]
                e += F.get("el", 0) * g
                for c in ("oil", "gas", "coal"):
                    v = F.get(c, 0) * g
                    if not v: continue
                    fuel[c] += v * (1 - z)
                    if z > 0: e += v * z * USEFUL[k][c] / ELEC[k]["ratio"]
            e += new_loads(scen["newLoadsTWh"], y) * 3.6 * r["elShare"]
            el.append(e * S["elX"][p])
        S["elPJ"].append(el); S["fuel"].append(fuel)
        S["loadMW"].append([e * MWH_PER_PJ / (1 - GRID_LOSS) / HOURS for e in el])
        tot = sum(S["loadMW"][p]); S["share"].append([x / tot for x in S["loadMW"][p]])
    S["only"] = list(range(NP)) if only is None else list(only)
    S["fixed"], S["budgetLeft"], S["fixedThrough"] = fixed, budget_left, fixed_through
    S["horizonEnd"] = NP - 1 if horizon_end is None else horizon_end
    return S

S0 = sys_of(DEFAULT)
print("z (share of fossil useful energy switched by 2050):", {k: round(v, 3) for k, v in S0["z"].items()})
print("electricity used, TWh:", [round(sum(e) / 3.6) for e in S0["elPJ"]], " · discount sums D:", [round(d, 3) for d in S0["D"]])

## 3. The pathway as one linear program

Units: power in MW (each slice's average), energy in PJ, CO₂ in Mt, money in millions of 2025 dollars ($M, present value in 2025).

**Decisions.** New wind, solar, batteries and gas plants in each region and model year (`n_…`), the generic firm clean option nation-wide (`n_firm…`: it has no location; its output is shared among the regions in proportion to their demand), link additions (`a_…`), the capacity of each existing coal, gas and oil unit kept open (`k_…`), what every plant makes in every slice, battery charge and discharge, flows, unserved energy, and the fuels bought.

**Goal.** min Σ (yearly capital and fixed O&M of new plants for the years they run up to 2050) + links + fixed O&M of fossil plants kept open + variable O&M + fuels bought + penalties + the tie rule. The reported cost leaves out the penalties and the tie rule and adds the existing wind, solar and batteries' fixed O&M while they are open (the new plants' coefficients): a constant, so it changes no decision. The data give no fixed O&M for the existing hydro and the steady plants.

**Rules, each region-slice-year.**

- balance: existing plants + new gas + hydro + wind + solar + battery out − in + share × firm + inflows × (1 − loss) − outflows + unserved = load
- each plant ≤ its availability × its capacity open (vintages: a plant built in model year q runs in model year p if p ≥ q and Y[p] − Y[q] < its life; q and p are indices 0 to 4, Y[p] their calendar years 2030 to 2050)
- batteries: the state at 08:00 and 20:00 between 0 and 4 h × power, cyclic each day; in the scarce-wind week the store may enter full and runs down over its seven days, and winter runs in order (its days, then the week, cyclic over the season), so winter's days pay the week back (energy in × 85% = energy out) and the level stays within the store
- links: flow one way + the other ≤ the link's limit

**Rules, each year.** Fuels (crude: final oil products ÷ 0.945, the refinery's yield, + the oil plants' fuel; gas: domestic fields, 5% of their gas used there, and LNG, in the least-cost mix within the import cap, so under a tight budget LNG can win because the national inventory counts the gas burned in domestic fields but not LNG's upstream emissions abroad; coal from Ridge County), imports ≤ cap × (primary energy − what batteries and links lose) (+ an overshoot; else wasting electricity would help meet the cap), the stress hours (each region's load × 1.1 met by plants at their availability, wind and solar at that hour's output, batteries at full power and links; a shortfall at $750,000 per MW-year; a drought derates the summer hour as it derates the summer slices), kept-open capacity never rises (a unit retired in an earlier window stays retired).

**The budget.** Σ weights × CO₂ of each model year ≤ the budget − 2 × 2025's (+ an overshoot).

**Limits first.** The budget and the import cap are limits, not prices. Each window is solved with the overshoots at their penalties ($1,000 a tonne, $100 a GJ); if it misses either, two more solves: the least overshoot (with no more unserved energy or stress-hour shortfall), then the least cost at that overshoot. `solve_window` does this, as the browser's engine does.

In [ ]:
class Model:
    def __init__(self):
        self.names, self.lb, self.ub, self.cost, self.idx, self.rows = [], [], [], [], {}, []
        self.cat, self.tie, self.vom, self.meta = {}, {}, {}, {}
    def var(self, name, lb=0.0, ub=math.inf, cost=0.0, cat=None, tie=0.0):
        self.idx[name] = len(self.names); self.names.append(name); self.lb.append(lb); self.ub.append(ub); self.cost.append(cost)
        if cat: self.cat[name] = cat
        if tie: self.tie[name] = tie
        return name
    def row(self, name, terms, lo=None, hi=None):
        self.rows.append((name, terms, lo, hi))
    def solve(self, cost=None, extra=(), upper=None):
        n = len(self.names); eq, ub, beq, bub, neq, nub = [], [], [], [], [], []
        for name, terms, lo, hi in list(self.rows) + list(extra):
            if lo is not None and hi is not None and lo == hi:
                eq.append(terms); beq.append(lo); neq.append(name)
            else:
                if hi is not None: ub.append(terms); bub.append(hi); nub.append((name, 1))
                if lo is not None: ub.append([(v, -c) for v, c in terms]); bub.append(-lo); nub.append((name, -1))
        def mat(rs):
            r, c, v = [], [], []
            for i, terms in enumerate(rs):
                for name, coef in terms: r.append(i); c.append(self.idx[name]); v.append(coef)
            return coo_matrix((v, (r, c)), shape=(len(rs), n)).tocsr()
        res = linprog(self.cost if cost is None else cost, A_ub=mat(ub) if ub else None, b_ub=bub or None, A_eq=mat(eq) if eq else None, b_eq=beq or None,
                      bounds=list(zip(self.lb, [None if u == math.inf else u for u in (self.ub if upper is None else upper)])), method="highs")
        out = dict(status=res.status, message=res.message, objective=res.fun)
        if res.status == 0:
            out["x"] = {nm: res.x[i] for i, nm in enumerate(self.names)}
            dual = {}
            for (nm, sgn), m in zip(nub, res.ineqlin.marginals): dual[nm] = dual.get(nm, 0.0) + sgn * m
            for nm, m in zip(neq, res.eqlin.marginals): dual[nm] = m
            out["dual"] = dual        # d objective / d right-hand side, as the browser's LX.solver reports it
        return out

def build(S, tie=1.0):
    pen, TIE, M = CFG["penalties"], CFG["tie"], Model()
    only, ft = S["only"], S["fixedThrough"]; inW = set(only)
    free = lambda q: q in inW and q > ft
    fx = S["fixed"] or dict(n={}, k={}, a={})
    lastP = S["horizonEnd"]
    served = lambda q, life: sum(S["D"][p] for p in range(q, lastP + 1) if Y[p] - Y[q] < life)
    online = lambda q, p, life: q <= p and Y[p] - Y[q] < life
    nv = {}
    for r in range(5):
        nv[r] = {}
        for t in NEW:
            row_ = []
            for q, y in enumerate(Y):
                nm = f"n_{r}{t}{q}"
                if not free(q): row_.append(fx["n"].get(nm, 0)); continue
                ann = S["capex"][q][t] * crf(S["rate"], T[t]["life"]) + T[t]["fom"]
                lim = LIM["gwPerRegionPerPeriod"][t] * 1000 if y >= LIM["availableFrom"] else 0
                row_.append(M.var(nm, 0, lim, ann * served(q, T[t]["life"]) / 1000, "capital"))
            nv[r][t] = row_
    fv = []
    for q, y in enumerate(Y):
        nm = f"n_firm{q}"
        if not free(q): fv.append(fx["n"].get(nm, 0)); continue
        ann = S["capex"][q]["firm"] * crf(S["rate"], T["firm"]["life"]) + T["firm"]["fom"]
        fv.append(M.var(nm, 0, LIM["gwPerRegionPerPeriod"]["firm"] * 1000 * 5 if S["firm"] else 0, ann * served(q, T["firm"]["life"]) / 1000, "capital"))
    av = []
    for l in LINKS:
        row_ = []
        for q in range(NP):
            nm = f"a_{l['i']}_{q}"
            if not free(q): row_.append(fx["a"].get(nm, 0)); continue
            row_.append(M.var(nm, 0, TX["maxAddMWPerPeriod"] if S["txOn"] else 0, l["km"] * S["txCost"] * crf(S["rate"], TX["life"]) * served(q, TX["life"]) / 1e6, "transmission"))
        av.append(row_)
    kv = []
    for u in UNITS:
        row_ = []
        for p, y in enumerate(Y):
            if not on(u, y): row_.append(0); continue
            nm = f"k_{u['j']}_{p}"
            if not free(p): row_.append(fx["k"].get(nm, u["mw"])); continue
            om = OM[u["tech"]]["fom"] if u["tech"] in ("coal", "oil") else T[u["tech"]]["fom"]
            row_.append(M.var(nm, 0 if S["retOn"] else u["mw"], u["mw"], om * (S["D"][p] if p <= lastP else 0) / 1000, "fom"))
        kv.append(row_)
    def cap_of(lst, p, life):
        terms, c = [], 0.0
        for q in range(p + 1):
            if online(q, p, life):
                if isinstance(lst[q], str): terms.append((lst[q], 1.0))
                else: c += lst[q]
        return terms, c
    CO2, PRIM, IMP = {}, {}, {}
    for p in only:
        y, D, pr, tg, sh = Y[p], S["D"][p], S["price"][p], f"_{p}", S["share"][p]
        bal = [[[] for _ in SL] for _ in range(5)]
        oilF, gasF, coalF, nucE, renE, vom, lossE = [], [], [], [], [], [], []
        for u in UNITS:
            k = kv[u["j"]][p]
            if not k: continue
            vr = OM[u["tech"]]["vom"] if u["tech"] in ("coal", "oil") else T[u["tech"]]["vom"]
            for s in SL:
                a = u["av"] * (S["summerX"][p] if s["summer"] and u["tech"] in ("coal", "gasCC") else 1)
                g = M.var(f"g_{u['j']}_{s['i']}{tg}", 0, math.inf if isinstance(k, str) else a * k, vr * s["h"] * D / 1e6, "vom")
                if isinstance(k, str): M.row(f"cu_{u['j']}_{s['i']}{tg}", [(g, 1), (k, -a)], None, 0)
                bal[u["r"]][s["i"]].append((g, 1)); vom.append((g, vr * s["h"] / 1e6))
                fl = s["h"] * PJ_PER_MWH / u["eff"]
                (coalF if u["tech"] == "coal" else oilF if u["tech"] == "oil" else gasF).append((g, fl))
        for r in range(5):
            nucMW, hyMW = exist("nuclear", r, y), exist("hydro", r, y)
            if nucMW > 0:
                for s in SL:
                    a = AV["nuclear"] * (S["summerX"][p] if s["summer"] else 1)
                    g = M.var(f"nu_{r}_{s['i']}{tg}", 0, a * nucMW, OM["nuclear"]["vom"] * s["h"] * D / 1e6, "vom")
                    bal[r][s["i"]].append((g, 1)); nucE.append((g, s["h"] * PJ_PER_MWH / 0.33)); vom.append((g, OM["nuclear"]["vom"] * s["h"] / 1e6))
            if hyMW > 0:
                for d in DAYS:
                    gd, gn = M.var(f"hy_{r}_{d['d']}{tg}", 0, hyMW), M.var(f"hy_{r}_{d['n']}{tg}", 0, hyMW)
                    e = 2 * hyMW * HYDRO_CF[r] * SL[d["d"]]["hydro"] * S["hydroX"][p]
                    M.row(f"he_{r}_{d['key']}{tg}", [(gd, 1), (gn, 1)], e, e)
                    bal[r][d["d"]].append((gd, 1)); bal[r][d["n"]].append((gn, 1))
                    renE += [(gd, SL[d["d"]]["h"] * PJ_PER_MWH), (gn, SL[d["n"]]["h"] * PJ_PER_MWH)]
            for t in ("wind", "solar"):
                terms, c = cap_of(nv[r][t], p, T[t]["life"]); c0 = c + exist(t, r, y)
                mx = (R[r]["res"]["windMaxGW"] if t == "wind" else R[r]["res"]["solarMaxGW"]) * 1000
                if terms: M.row(f"mx_{r}{t}{tg}", terms, None, mx - c0)
                for s in SL:
                    cf = s["wind"][r] if t == "wind" else s["solar"][r]
                    g = M.var(f"vr_{r}{t}_{s['i']}{tg}", 0, math.inf if terms else cf * c0)
                    if terms: M.row(f"cv_{r}{t}_{s['i']}{tg}", [(g, 1)] + [(x, -cf) for x, _ in terms], None, cf * c0)
                    bal[r][s["i"]].append((g, 1)); renE.append((g, s["h"] * PJ_PER_MWH))
            for t in ("gasCC", "gasCT"):
                terms, c = cap_of(nv[r][t], p, T[t]["life"])
                if not terms and c == 0: continue
                for s in SL:
                    a = AV[t] * (S["summerX"][p] if s["summer"] and t == "gasCC" else 1)
                    g = M.var(f"ng_{r}{t}_{s['i']}{tg}", 0, math.inf if terms else a * c, T[t]["vom"] * s["h"] * D / 1e6, "vom")
                    if terms: M.row(f"cn_{r}{t}_{s['i']}{tg}", [(g, 1)] + [(x, -a) for x, _ in terms], None, a * c)
                    bal[r][s["i"]].append((g, 1)); gasF.append((g, s["h"] * PJ_PER_MWH / T[t]["efficiency"])); vom.append((g, T[t]["vom"] * s["h"] / 1e6))
            bt, bcst = cap_of(nv[r]["battery"], p, T["battery"]["life"]); b0 = bcst + exist("battery", r, y)
            if bt or b0 > 0:
                hrs, eta = T["battery"]["hours"], T["battery"]["roundTrip"]; ch, dis = {}, {}
                for s in SL:
                    wear = TIE["batteryPerMWh"] * tie * s["h"] * D / 1e6
                    ch[s["i"]] = M.var(f"bc_{r}_{s['i']}{tg}"); dis[s["i"]] = M.var(f"bd_{r}_{s['i']}{tg}", 0, math.inf, wear, None, wear)
                    M.row(f"bp_{r}_{s['i']}{tg}", [(ch[s["i"]], 1), (dis[s["i"]], 1)] + [(x, -1) for x, _ in bt], None, b0)
                    bal[r][s["i"]] += [(dis[s["i"]], 1), (ch[s["i"]], -1)]
                E = lambda extra: extra + [(x, -hrs) for x, _ in bt]          # terms − 4 × power
                for d in DAYS:
                    s0 = M.var(f"bs_{r}_{d['key']}{tg}"); hd = SL[d["d"]]["h"] / SL[d["d"]]["days"]; hn = SL[d["n"]]["h"] / SL[d["n"]]["days"]
                    day_end = [(s0, 1), (ch[d["d"]], hd * eta), (dis[d["d"]], -hd)]
                    M.row(f"b0_{r}_{d['key']}{tg}", E([(s0, 1)]), None, hrs * b0)
                    M.row(f"b1_{r}_{d['key']}{tg}", day_end, 0, None)
                    M.row(f"b2_{r}_{d['key']}{tg}", E(day_end), None, hrs * b0)
                    delta = [(ch[d["d"]], hd * eta), (dis[d["d"]], -hd), (ch[d["n"]], hn * eta), (dis[d["n"]], -hn)]
                    if d is REFILL:                       # winter in order: its days, then the week, cyclic over the season
                        hdw = SL[WEEK["d"]]["h"] / WEEK["days"]; hnw = SL[WEEK["n"]]["h"] / WEEK["days"]; K = d["days"]; sw = f"bs_{r}_{WEEK['key']}{tg}"
                        dw = [(ch[WEEK["d"]], hdw * eta), (dis[WEEK["d"]], -hdw), (ch[WEEK["n"]], hnw * eta), (dis[WEEK["n"]], -hnw)]
                        at = lambda k: [(s0, 1)] + [(v, c * k) for v, c in delta]                # the level at the start of winter day k
                        last_start = at(K - 1); last_noon = last_start + [(ch[d["d"]], hd * eta), (dis[d["d"]], -hd)]
                        M.row(f"b3_{r}_{d['key']}{tg}", last_start, 0, None); M.row(f"b4_{r}_{d['key']}{tg}", E(last_start), None, hrs * b0)
                        M.row(f"b5_{r}_{d['key']}{tg}", last_noon, 0, None); M.row(f"b6_{r}_{d['key']}{tg}", E(last_noon), None, hrs * b0)
                        M.row(f"bk_{r}_{d['key']}{tg}", [(sw, 1)] + [(v, -c) for v, c in at(K)], 0, 0)               # the week starts where winter's days end
                        M.row(f"bz_{r}_{d['key']}{tg}", [(sw, 1)] + [(v, c * WEEK["days"]) for v, c in dw] + [(s0, -1)], 0, 0)   # and ends where winter began
                    elif not d["week"]:
                        M.row(f"bz_{r}_{d['key']}{tg}", delta, 0, 0)
                    else:
                        N = d["days"]; times = lambda k: [(v, c * k) for v, c in delta]
                        end_w = [(s0, 1)] + times(N); last_day = [(s0, 1)] + times(N - 1) + [(ch[d["d"]], hd * eta), (dis[d["d"]], -hd)]
                        M.row(f"bw0_{r}{tg}", end_w, 0, None); M.row(f"bw1_{r}{tg}", E(end_w), None, hrs * b0)
                        M.row(f"bw2_{r}{tg}", last_day, 0, None); M.row(f"bw3_{r}{tg}", E(last_day), None, hrs * b0)
                for s in SL: lossE += [(ch[s["i"]], s["h"] * PJ_PER_MWH), (dis[s["i"]], -s["h"] * PJ_PER_MWH)]   # what the store loses
            for s in SL:
                us = M.var(f"us_{r}_{s['i']}{tg}", 0, math.inf, pen["unservedPerMWh"] * s["h"] * D / 1e6, "penalty"); bal[r][s["i"]].append((us, 1))
        fterms, fc = cap_of(fv, p, T["firm"]["life"])
        if fterms or fc > 0:
            for s in SL:
                a = T["firm"]["availability"] * (S["summerX"][p] if s["summer"] else 1)
                g = M.var(f"fg_{s['i']}{tg}", 0, math.inf if fterms else a * fc, T["firm"]["vom"] * s["h"] * D / 1e6, "vom")
                if fterms: M.row(f"cf_{s['i']}{tg}", [(g, 1)] + [(x, -a) for x, _ in fterms], None, a * fc)
                for r in range(5): bal[r][s["i"]].append((g, sh[r]))
                renE.append((g, s["h"] * PJ_PER_MWH)); vom.append((g, T["firm"]["vom"] * s["h"] / 1e6))
        for l in LINKS:
            terms, c = cap_of(av[l["i"]], p, 1e9); c0 = c + l["mw"]
            for s in SL:
                cc = TIE["flowPerMWh"] * tie * s["h"] * D / 1e6
                fw = M.var(f"fw_{l['i']}_{s['i']}{tg}", 0, math.inf, cc, None, cc); bw = M.var(f"fb_{l['i']}_{s['i']}{tg}", 0, math.inf, cc, None, cc)
                M.row(f"cl_{l['i']}_{s['i']}{tg}", [(fw, 1), (bw, 1)] + [(x, -1) for x, _ in terms], None, c0)
                bal[l["a"]][s["i"]] += [(fw, -1), (bw, 1 - l["loss"])]; bal[l["b"]][s["i"]] += [(fw, 1 - l["loss"]), (bw, -1)]
                lossE += [(fw, l["loss"] * s["h"] * PJ_PER_MWH), (bw, l["loss"] * s["h"] * PJ_PER_MWH)]          # what the link loses
        if S["reserve"] is not None:                         # the peak-hour reserve rule at the stress hours
            for hi, H in enumerate(CFG["stress"]):
                sx = S["summerX"][p] if H["season"] == "summer" else 1.0          # a drought derates the summer hour
                dr = lambda t: sx if t in ("coal", "gasCC") else 1.0
                sb, c0 = [[] for _ in range(5)], [0.0] * 5
                for u in UNITS:
                    k = kv[u["j"]][p]
                    if not k: continue
                    if isinstance(k, str): sb[u["r"]].append((k, u["av"] * dr(u["tech"])))
                    else: c0[u["r"]] += u["av"] * dr(u["tech"]) * k
                for r in range(5):
                    c0[r] += AV["nuclear"] * sx * exist("nuclear", r, y) + exist("hydro", r, y) * AV["hydro"] * S["hydroX"][p]
                    for t in ("wind", "solar"):
                        terms, c = cap_of(nv[r][t], p, T[t]["life"]); f = H[t][r]
                        sb[r] += [(x, f) for x, _ in terms]; c0[r] += f * (c + exist(t, r, y))
                    for t in ("gasCC", "gasCT"):
                        terms, c = cap_of(nv[r][t], p, T[t]["life"]); a = AV[t] * dr(t)
                        sb[r] += [(x, a) for x, _ in terms]; c0[r] += a * c
                    terms, c = cap_of(nv[r]["battery"], p, T["battery"]["life"]); sb[r] += terms; c0[r] += c + exist("battery", r, y)
                    terms, c = cap_of(fv, p, T["firm"]["life"]); a = T["firm"]["availability"] * sx * sh[r]
                    sb[r] += [(x, a) for x, _ in terms]; c0[r] += a * c
                for l in LINKS:
                    terms, c = cap_of(av[l["i"]], p, 1e9)
                    fw, bw = M.var(f"sw_{hi}_{l['i']}{tg}"), M.var(f"sx_{hi}_{l['i']}{tg}")
                    M.row(f"sl_{hi}_{l['i']}{tg}", [(fw, 1), (bw, 1)] + [(x, -1) for x, _ in terms], None, c + l["mw"])
                    sb[l["a"]] += [(fw, -1), (bw, 1 - l["loss"])]; sb[l["b"]] += [(fw, 1 - l["loss"]), (bw, -1)]
                for r in range(5):
                    sf = M.var(f"ss_{hi}_{r}{tg}", 0, math.inf, pen["reservePerMWYear"] * D / 1e6, "penalty"); sb[r].append((sf, 1))
                    M.row(f"st_{hi}_{r}{tg}", sb[r], S["loadMW"][p][r] * H["demand"] * (1 + S["reserve"]) - c0[r], None)
        for r in range(5):
            for s in SL:
                Ld = S["loadMW"][p][r] * s["demand"]; M.row(f"bal_{r}_{s['i']}{tg}", bal[r][s["i"]], Ld, Ld)
        F = S["fuel"][p]; prem = TIE["importPremiumPerGJ"] * tie * D
        cd = M.var("cdom" + tg, 0, DOM_CRUDE, pr["oil"] * D, "fuel"); ci = M.var("cimp" + tg, 0, math.inf, (pr["oil"] + TIE["importPremiumPerGJ"] * tie) * D, "fuel", prem)
        gd = M.var("gdom" + tg, 0, DOM_GAS, pr["gas"] * D, "fuel"); gl = M.var("lng" + tg, 0, math.inf, pr["lng"] * D, "fuel"); qc = M.var("coal" + tg, 0, COAL_MAX, pr["coal"] * D, "fuel")
        M.row("oil" + tg, [(cd, 1), (ci, 1)] + [(x, -c) for x, c in oilF], F["oil"] / REF_YIELD, F["oil"] / REF_YIELD)
        M.row("gas" + tg, [(gd, 1 - GAS_OWN), (gl, 1)] + [(x, -c) for x, c in gasF], F["gas"], F["gas"])
        M.row("cl" + tg, [(qc, 1 - COAL_OWN)] + [(x, -c) for x, c in coalF], F["coal"], F["coal"])
        CO2[p] = [(cd, FAC["oil"] / 1000), (ci, FAC["oil"] / 1000), (gd, FAC["gas"] / 1000), (gl, FAC["gas"] / 1000), (qc, FAC["coal"] / 1000)]
        PRIM[p] = [(cd, 1), (ci, 1), (gd, 1), (gl, 1), (qc, 1)] + nucE + renE
        IMP[p] = [(ci, 1), (gl, 1)]
        oi = M.var("oimp" + tg, 0, math.inf, pen["importPerGJ"], "penalty")
        t = {}
        for x, c in IMP[p]: t[x] = t.get(x, 0) + c
        for x, c in PRIM[p]: t[x] = t.get(x, 0) - S["cap"] * c
        for x, c in lossE: t[x] = t.get(x, 0) + S["cap"] * c      # the cap's base leaves out what batteries and links lose
        t[oi] = t.get(oi, 0) - 1
        M.row("imp" + tg, list(t.items()), None, 0)
        M.vom[p] = vom
    if S["retOn"]:                                           # retirements never reverse, across windows too
        for u in UNITS:
            for p in range(1, NP):
                a, b = kv[u["j"]][p - 1], kv[u["j"]][p]
                if not isinstance(b, str) or not on(u, Y[p - 1]): continue
                M.row(f"kr_{u['j']}_{p}", [(b, 1), (a, -1)] if isinstance(a, str) else [(b, 1)], None, 0 if isinstance(a, str) else a)
    left = S["budgetLeft"] if S["budgetLeft"] is not None else S["budget"] - S["w0"] * PL["carbon"]["base2025Mt"]
    ob = M.var("over", 0, math.inf, pen["budgetPerT"], "penalty")
    M.row("budget", [(ob, -1)] + [(x, c * S["w"][p]) for p in only for x, c in CO2[p]], None, left)
    M.meta = dict(CO2=CO2, PRIM=PRIM, IMP=IMP, left=left)
    return M

M0 = build(S0)
print(f"the central plan: {len(M0.names):,} decisions, {len(M0.rows):,} rules")

## 4. Solve the central plan, and check it against the browser

`read` takes the decisions of every model year and what ran in each; `money` gives each model year's yearly cost (new plants' payments and fixed O&M for what is open, links, fixed O&M of fossil plants kept open, variable O&M, every fuel bought) and its present value, Σ D[p] × yearly[p]. The penalties and the tie rule are never part of a cost. `run(plan)` does it all, for perfect foresight or decade by decade (section 9 and practice 2).

In [ ]:
def read(S, M, r):
    x, du, fx = r["x"], r["dual"], S["fixed"] or dict(n={}, k={}, a={})
    val = lambda n: x.get(n, 0.0)
    dec = lambda nm, d: x[nm] if nm in x else fx[nm[0]].get(nm, d)
    raw = dict(only=list(S["only"]), per={}, over=val("over"), budgetDual=du.get("budget", 0.0), reach=r.get("reach", True),
               n=[{t: [dec(f"n_{rr}{t}{q}", 0) for q in range(NP)] for t in NEW} for rr in range(5)],
               firm=[dec(f"n_firm{q}", 0) for q in range(NP)],
               a=[[dec(f"a_{l['i']}_{q}", 0) for q in range(NP)] for l in LINKS],
               k=[[dec(f"k_{u['j']}_{p}", u["mw"]) if on(u, Y[p]) else 0 for p in range(NP)] for u in UNITS])
    for p in S["only"]:
        tg = f"_{p}"; lin = lambda terms: sum(val(n) * c for n, c in terms)
        q = dict(fuel=dict(crudeDom=val("cdom" + tg), crudeImp=val("cimp" + tg), gasDom=val("gdom" + tg), lng=val("lng" + tg), coal=val("coal" + tg)),
                 co2=lin(M.meta["CO2"][p]), primary=lin(M.meta["PRIM"][p]), imports=lin(M.meta["IMP"][p]), vom=lin(M.vom[p]),
                 unserved=sum(val(f"us_{rr}_{s['i']}{tg}") * s["h"] for rr in range(5) for s in SL) / 1e6,
                 stressShort=[sum(val(f"ss_{hi}_{rr}{tg}") for rr in range(5)) for hi in range(len(CFG["stress"]))])
        q["importShare"] = q["imports"] / q["primary"]
        raw["per"][p] = q
    return raw

UPKEEP = ("wind", "solar", "battery")      # existing units at the new plants' fixed O&M, while open (a constant)

def upkeep(y):
    return sum(exist(t, r, y) * T[t]["fom"] / 1000 for t in UPKEEP for r in range(5))   # $M a year

def money(S, raw):
    yearly = []
    for p, y in enumerate(Y):
        o = dict(capital=0.0, transmission=0.0, fom=0.0, upkeep=upkeep(y), vom=raw["per"][p]["vom"], fuel=0.0)
        for rr in range(5):
            for t in NEW:
                for b in range(p + 1):
                    if Y[p] - Y[b] < T[t]["life"]: o["capital"] += raw["n"][rr][t][b] * (S["capex"][b][t] * crf(S["rate"], T[t]["life"]) + T[t]["fom"]) / 1000
        for b in range(p + 1):
            if Y[p] - Y[b] < T["firm"]["life"]: o["capital"] += raw["firm"][b] * (S["capex"][b]["firm"] * crf(S["rate"], T["firm"]["life"]) + T["firm"]["fom"]) / 1000
        for l in LINKS:
            o["transmission"] += sum(raw["a"][l["i"]][b] for b in range(p + 1)) * l["km"] * S["txCost"] * crf(S["rate"], TX["life"]) / 1e6
        for u in UNITS:
            om = OM[u["tech"]]["fom"] if u["tech"] in ("coal", "oil") else T[u["tech"]]["fom"]
            o["fom"] += raw["k"][u["j"]][p] * om / 1000
        f, pr = raw["per"][p]["fuel"], S["price"][p]
        o["fuel"] = (f["crudeDom"] + f["crudeImp"]) * pr["oil"] + f["gasDom"] * pr["gas"] + f["lng"] * pr["lng"] + f["coal"] * pr["coal"]
        o["total"] = sum(o.values())
        yearly.append(o)
    return yearly, sum(S["D"][p] * yearly[p]["total"] for p in range(NP))

def summary(S, raw):
    yearly, pv = money(S, raw)
    co2 = [raw["per"][p]["co2"] for p in range(NP)]
    built = {t: [sum(raw["n"][rr][t][q] for rr in range(5)) / 1000 for q in range(NP)] for t in NEW}
    built["firm"] = [v / 1000 for v in raw["firm"]]
    dv = raw.get("duals", [raw["budgetDual"] if raw.get("reach", True) else None])
    return dict(pv=pv, yearly=yearly, co2=co2, cum=S["w0"] * PL["carbon"]["base2025Mt"] + sum(w * c for w, c in zip(S["w"], co2)),
                imp=[raw["per"][p]["importShare"] for p in range(NP)], built=built, carbonValue=[None if d is None else -d for d in dv], over=raw["over"],
                unserved=[raw["per"][p]["unserved"] for p in range(NP)], fuel=[raw["per"][p]["fuel"] for p in range(NP)],
                reserveShort=[max(raw["per"][p]["stressShort"], default=0.0) for p in range(NP)], raw=raw, S=S)

def windows_of(fs, frm=0):
    allw = [[0, 1], [2, 3], [4]] if fs == "myopic" else [[0, 1, 2, 3, 4]]
    return [w2 for w2 in ([p for p in w if p >= frm] for w in allw) if w2]

def fixed_of(raw, f):
    o = dict(n={}, k={}, a={})
    for q in range(f + 1):
        for rr in range(5):
            for t in NEW: o["n"][f"n_{rr}{t}{q}"] = raw["n"][rr][t][q]
        o["n"][f"n_firm{q}"] = raw["firm"][q]
        for l in LINKS: o["a"][f"a_{l['i']}_{q}"] = raw["a"][l["i"]][q]
        for u in UNITS:
            if on(u, Y[q]): o["k"][f"k_{u['j']}_{q}"] = raw["k"][u["j"]][q]
    return o

LIM_TOL = dict(miss=0.01, svc=0.001)   # $M, as the browser's engine: 10 t over the budget or 0.1 TJ over the cap; $1,000 of penalties

def solve_window(M):
    # limits first: the LP with the overshoots at their penalties; if it misses the budget or the cap, the least
    # overshoot (no more unserved energy or stress-hour shortfall than that run), then the least cost at that overshoot
    r = M.solve()
    if r["status"] != 0: return r
    MT = [(n, M.cost[M.idx[n]]) for n in M.names if n == "over" or n.startswith("oimp_")]
    ST = [(n, M.cost[M.idx[n]]) for n in M.names if n.startswith(("us_", "ss_"))]
    lin = lambda T_, x: sum(x[n] * c for n, c in T_)
    r["reach"] = True
    if lin(MT, r["x"]) <= LIM_TOL["miss"]: return r
    svc = lin(ST, r["x"]); service = [("service", ST, None, svc + max(LIM_TOL["svc"], 1e-7 * svc))]
    c2 = np.zeros(len(M.names))
    for n, c in MT: c2[M.idx[n]] = c
    r2 = M.solve(cost=c2, extra=service)
    if r2["status"] != 0: return r2
    reach = r2["objective"] <= LIM_TOL["miss"]
    def third(hard):
        ub, c3 = list(M.ub), list(M.cost)
        for n, _ in MT:
            i = M.idx[n]; ub[i] = 0.0 if hard else r2["x"][n] * (1 + 1e-9) + 1e-6; c3[i] = 0.0
        return M.solve(cost=c3, extra=service, upper=ub)
    r3 = third(reach)
    if reach and r3["status"] == 2: r3 = third(False)        # numerically infeasible at exactly 0: allow run 2's overshoot
    if r3["status"] != 0: return r3
    r3["reach"] = reach
    return r3

def run_windows(plan, shock=None, frm=0, fixed=None, fixed_through=-1, spent=0.0, tie=1.0):
    S0_ = sys_of(plan, shock=shock); spent += S0_["w0"] * PL["carbon"]["base2025Mt"]
    wins = windows_of(plan["fs"], frm); fixed = fixed or dict(n={}, k={}, a={}); raws = []
    for i, win in enumerate(wins):
        rest = [p for w in wins[i:] for p in w]
        left = (plan["bud"] - spent) * sum(S0_["w"][p] for p in win) / sum(S0_["w"][p] for p in rest)
        S = sys_of(plan, shock=shock, only=win, fixed=fixed, budget_left=left, horizon_end=win[-1] if plan["fs"] == "myopic" else NP - 1, fixed_through=fixed_through)
        M = build(S, tie); r = solve_window(M)
        assert r["status"] == 0, r["message"]
        raw = read(S, M, r); raws.append(raw)
        spent += sum(S["w"][p] * raw["per"][p]["co2"] for p in win)
        fixed = fixed_of(raw, win[-1])
    out = copy.deepcopy(raws[-1])                            # decisions: the last window holds every earlier one as fixed
    out["per"] = {p: next(rw["per"][p] for rw in raws if p in rw["per"]) for p in range(NP) if any(p in rw["per"] for rw in raws)}
    out["over"] = sum(rw["over"] for rw in raws); out["duals"] = [rw["budgetDual"] if rw["reach"] else None for rw in raws]
    return out

def run(plan, tie=1.0):
    # the plan's pathway, without a shock: a shock's re-plan is practice task 3 (run_windows with shock=...)
    assert plan.get("sh", "none") == "none", "run() solves the plan without a shock; set sh='none' (practice task 3 runs a shock's re-plan)"
    return summary(sys_of(plan), run_windows(plan, tie=tie))

def lib(preset, foresight="perfect"):
    return next(e for e in LIB["plans"] if e["preset"] == preset and e["foresight"] == foresight)

def check(O, V, what):
    assert abs(O["pv"] - V["pv"]["total"]) < 5, (what, "cost", O["pv"], V["pv"]["total"])
    assert abs(O["cum"] - V["cum"]) < 1, (what, "cumulative CO2", O["cum"], V["cum"])
    assert all(abs(a - b) < 0.1 for a, b in zip(O["co2"], V["co2"])), (what, "CO2", O["co2"], V["co2"])
    assert all(abs(a - b) < 0.002 for a, b in zip(O["imp"], V["imp"])), (what, "imports", O["imp"], V["imp"])
    for t, v in V["built"].items():
        assert all(abs(a - b) < 0.05 for a, b in zip(O["built"][t], v)), (what, "built " + t, O["built"][t], v)
    for a, b in zip(O["carbonValue"], V["carbonValue"]):
        assert (a is None) == (b is None) and (b is None or abs(a - b) <= 0.01 * abs(b) + 0.06), (what, "carbon value", a, b)
    print("ok:", what, "matches the browser's")

def show(name, O):
    print(f"{name}: cost to 2050 ${O['pv'] / 1000:,.1f} billion (present value); CO2 2026-2050 {O['cum']:,.0f} Mt"
          + (f" ({O['over']:,.0f} Mt over the budget)" if O["over"] > 0.5 else ""))
    print("  CO2 a year, Mt:", [round(float(c), 1) for c in O["co2"]], "· imports:", [f"{v:.1%}" for v in O["imp"]])
    print("  built, GW:", {t: round(float(sum(v)), 1) for t, v in O["built"].items()})

CENTRAL = run(DEFAULT)
show("central (perfect foresight)", CENTRAL)
check(CENTRAL, lib("central")["view"], "the central plan")
check(run(PRESET["secure"]), lib("secure")["view"], "the security-first plan (its import cap out of reach in 2030)")

## 5. The budget's dual, and its price in each model year

The budget row's dual is what one more tonne allowed in 2026–2050 would save, in 2025's present value: a marginal value under this model's assumptions, not a market price.

- **Its price in a model year.** The goal counts a model year's yearly costs `D[p]` times; the budget row counts its CO₂ `w[p]` times. So one more tonne a year in model year `p` is worth dual × `w[p]` ÷ `D[p]`, in that year's 2025 dollars.
- **Hotelling's rule.** In a model of every year, each counted once, that price would rise at the discount rate: dual × (1 + rate)^(t − 2025).
- **Why they differ here.** Straight-line counting gives 2050 three years of CO₂ against five years of costs, so its price drops; with `acc="step"` the price rises at the discount rate between model years.

In [ ]:
cv = CENTRAL["carbonValue"][0]; rate = DEFAULT["dr"] / 100; SD = sys_of(DEFAULT)
print(f"the budget's dual: ${cv:,.1f} a tonne, present value in 2025")
for p, y in enumerate(Y):
    print(f"  {y}: the LP's price, dual x w / D = ${cv * SD['w'][p] / SD['D'][p]:,.0f} a tonne; Hotelling's every-year path: ${cv * (1 + rate) ** (y - 2025):,.0f}")
assert all(abs(cv * SD["w"][p] / SD["D"][p] - lib("central")["view"]["carbonPrice"][p]) < 0.05 * SD["w"][p] / SD["D"][p] + 0.2 for p in range(NP)), "the browser's prices"
# the dual is the cost slope: move the budget 10 Mt either way
up, dn = run(dict(DEFAULT, bud=DEFAULT["bud"] + 10)), run(dict(DEFAULT, bud=DEFAULT["bud"] - 10))
slope = (dn["pv"] - up["pv"]) / 20
print(f"cost slope over ±10 Mt: ${slope:,.1f} a tonne")
assert abs(slope - cv) < 0.02 * cv + 0.05

## 6. Deeper cuts, and a budget out of reach

The page's "deep cuts" preset electrifies as far as the data allows (buildings 100%, transport 78%, industry 60% of useful energy by 2050) with a 3,000 Mt budget.

- **Out of reach.** With the central plan's electrification, 3,000 Mt is out of reach: the fuels that stay (oil for vehicles and farms, gas and coal for industry's heat) emit too much, whatever the power sector does.
- **The closest pathway.** `solve_window` then finds the least CO₂ these levers allow (serving every slice and stress hour), and the least cost at that level; the budget has no dual there.

In [ ]:
DEEP = run(PRESET["deep"]); show("deep cuts", DEEP)
check(DEEP, lib("deep")["view"], "the deep-cuts plan")
TIGHT = run(dict(DEFAULT, bud=3000)); show("central electrification, 3,000 Mt", TIGHT)
assert TIGHT["over"] > 100, "with the central plan's electrification, 3,000 Mt is out of reach"

## 7. Without the peak-hour rule

The slices are twelve-hour averages: a plan that only meets them can lean on wind and solar at hours when they are not there. The peak-hour rule asks every region to meet its load at two stress hours of 2025 (the peak net-load hour and the busiest hour of the scarce-wind week) with 10% to spare. Turn it off and see what the cheapest pathway keeps open. (In the lesson, the full-year check runs both plans hour by hour: without the rule, demand goes unserved in dozens of hours.)

In [ ]:
OFF = run(dict(DEFAULT, rm="off")); show("no peak-hour rule", OFF)
assert OFF["pv"] <= CENTRAL["pv"] + 1, "removing a rule never raises the least cost"
early_on = sum(u["mw"] - CENTRAL["raw"]["k"][u["j"]][0] for u in UNITS if on(u, 2030)) / 1000
early_off = sum(u["mw"] - OFF["raw"]["k"][u["j"]][0] for u in UNITS if on(u, 2030)) / 1000
gas_on, gas_off = sum(CENTRAL["built"]["gasCC"]) + sum(CENTRAL["built"]["gasCT"]), sum(OFF["built"]["gasCC"]) + sum(OFF["built"]["gasCT"])
print(f"fossil retired early by 2030: {early_on:.1f} GW with the rule, {early_off:.1f} GW without; new gas: {gas_on:.1f} GW with, {gas_off:.1f} GW without")

## 8. Your pathway (capstone C4)

Set your C4 plan in the cell below (the one that starts `MY = `; the keys are the Pathway Explorer's, listed in `DEFAULT`) and run it. Hand in the printed lines with your policy brief. The link opens the same plan in the Pathway Explorer.

In [ ]:
ORDER = [("dem", "d"), ("eb", "eb"), ("et", "et"), ("ei", "ei"), ("bud", "b"), ("acc", "a"), ("cap", "c"), ("fs", "f"), ("rm", "m"), ("fm", "fm"), ("tx", "t"), ("rt", "r"),
         ("sh", "s"), ("cw", "cw"), ("cs", "cs"), ("cb", "cb"), ("cc", "cc"), ("ct", "ct"), ("cf", "cf"), ("cx", "cx"), ("pg", "pg"), ("pl", "pl"), ("pc", "pc"), ("po", "po"), ("dr", "dr")]
ENUM = dict(dem=[x["key"] for x in PL["demand"]["scenarios"]], acc=["linear", "step"], fs=["perfect", "myopic"], rm=["off", "r0", "r10", "r20"],
            sh=["none", "gasSpike", "drought", "mineralBan", "demandSurprise"])
def num(v):
    s = f"{v:.2f}".rstrip("0").rstrip(".") if isinstance(v, float) else str(v)
    return s.replace("-", "_").replace(".", "d")
def explorer_link(p):
    return "https://henryutd2021.github.io/learn/sb4-pathway-explorer.html#sb4-v1-" + "-".join(sh + (str(ENUM[k].index(p[k])) if k in ENUM else num(p[k])) for k, sh in ORDER)

MY = dict(DEFAULT, eb=90, et=65, ei=60, bud=3400, cap=35)   # your levers
O = run(MY); show("your pathway", O)
print("the same plan in the Pathway Explorer:", explorer_link(MY))

## Practice (about 120 minutes)

Each task has a worked answer in the cell after it; try the task first.

1. **The straight-line budget.** prairieland.json's preset "a straight line to zero in 2050" is a 2,579 Mt budget: the CO₂ a straight-line decline from 2025's level to zero in 2050 would emit. It caps only the total; the yearly path and the 2050 level are free. Run it with the most electrification the data allows (`PRESET["deep"]` with `bud=2579`). By how much does the best pathway miss it? Which fuel uses are left in 2050, and why can this model not cut them?
2. **Decade by decade.** Run the central plan with `fs="myopic"`: 2030–2035 are planned seeing only themselves, their builds are fixed, then 2040–2045, then 2050; each window gets the budget left in proportion to its weights. Compare the cost and the CO₂ path with perfect foresight. Why must the myopic pathway cost at least as much?
3. **A demand surprise.** Electricity demand turns out 10% higher from 2035, unknown when 2030 and 2035 were planned. Keep the central plan's builds and retirements up to and including 2035, run 2035 with what was built, and re-plan 2040–2050 knowing it. What does the surprise cost, and where does the plan miss the peak-hour rule?
4. **Near-optimal plans.** Among all plans within one millionth of the least cost (the economic cost, without the tie rule's small costs), how much new solar can be built in the 2050 model year (2046–2050), at least and at most? What does the width of that range say about the model's "answer"?
5. **Counting the years between.** Switch the budget's weights to `acc="step"` (each model year for the five years ending in it). Why does the same 3,600 Mt then allow more CO₂ in 2026–2029, and which accounting is closer to what the atmosphere sees?

In [ ]:
# worked answer 1: the straight-line budget (2,579 Mt in all) is out of reach
LINE = run(dict(PRESET["deep"], bud=2579)); show("deep cuts, 2,579 Mt", LINE)
f50 = sys_of(PRESET["deep"])["fuel"][-1]
print(f"2050 final fuels (fixed by the plan, not the model): oil {f50['oil']:,.0f} PJ, gas {f50['gas']:,.0f} PJ, coal {f50['coal']:,.0f} PJ")
nonpower = (f50["oil"] / REF_YIELD * FAC["oil"] + f50["gas"] * FAC["gas"] + f50["coal"] * FAC["coal"]) / 1000
print(f"their CO2 alone in 2050: about {nonpower:,.0f} Mt a year")
assert LINE["over"] > 50 and nonpower > 40
print("answer: the closest pathway misses the budget by about", round(LINE["over"]), "Mt (no pathway emits less); transport and farm oil and industry's gas and coal remain, and the model has no clean fuel for them")

In [ ]:
# worked answer 2: decade by decade
MYO = run(dict(DEFAULT, fs="myopic")); show("central, decade by decade", MYO)
check(MYO, lib("central", "myopic")["view"], "the central plan, decade by decade")
assert MYO["pv"] >= CENTRAL["pv"] - 1
print(f"this decade-by-decade rule (two model years a window, each given its share of the budget left) costs ${(MYO['pv'] - CENTRAL['pv']) / 1000:,.1f} billion more; window carbon values: {[round(float(v), 1) for v in MYO['carbonValue']]} $/t")
print("answer: perfect foresight could have chosen the myopic pathway (it meets the same rules), so its least cost cannot be higher")

In [ ]:
# worked answer 3: a demand surprise from 2035, builds up to 2035 kept
plan_raw = CENTRAL["raw"]; Ss = sys_of(DEFAULT, shock="demandSurprise"); sp = Ss["shockP"]
spent = sum(Ss["w"][p] * plan_raw["per"][p]["co2"] for p in range(sp))
re = run_windows(DEFAULT, shock="demandSurprise", frm=sp, fixed=fixed_of(plan_raw, sp), fixed_through=sp, spent=spent)
for p in range(sp): re["per"][p] = plan_raw["per"][p]
SUR = summary(Ss, re); show("central, after a demand surprise", SUR)
X = lib("central")["shocks"]["demandSurprise"]
assert abs(SUR["pv"] - X["pv"]) < 5 and abs(SUR["cum"] - X["cum"]) < 1 and all(abs(a - b) < 0.1 for a, b in zip(SUR["co2"], X["co2"]))
print(f"the surprise costs ${(SUR['pv'] - CENTRAL['pv']) / 1000:,.1f} billion more (present value); imports in 2035: {SUR['imp'][sp]:.1%} against {CENTRAL['imp'][sp]:.1%}")
short = SUR["reserveShort"][sp]                              # this solution's own stress-hour shortfall, MW
assert abs(short - X["reserveShort"][sp]) <= 2, (short, X["reserveShort"][sp])
print(f"at its worse stress hour in 2035 the re-plan is {short:,.0f} MW short of the peak-hour rule, which asks for each region's load + 10%:")
print("the pathway misses its 10% reserve requirement; that alone does not show that peak demand goes unserved")
print("ok: the re-plan matches the browser's, its shortfall too")

In [ ]:
# worked answer 4: near-optimal plans (a millionth of the least cost)
M = build(S0); r = M.solve()                                     # the plan shown (with the tie rule)
econ = [c - M.tie.get(n, 0.0) for n, c in zip(M.names, M.cost)]  # the economic cost: the goal without the tie rule
best = M.solve(cost=econ)["objective"]                             # the least economic cost
slack = 1e-6 * (best + sum(S0["D"][p] * upkeep(y) for p, y in enumerate(Y)))   # a millionth of it, with the existing fleet's upkeep
face = [("face", [(n, c) for n, c in zip(M.names, econ) if c], None, best + slack)]
target = [f"n_{rr}solar4" for rr in range(5)]
cvec = np.zeros(len(M.names))
for n in target: cvec[M.idx[n]] = 1.0
lo = M.solve(cost=cvec, extra=face)["objective"] / 1000
hi = -M.solve(cost=-cvec, extra=face)["objective"] / 1000
plan50 = CENTRAL["built"]["solar"][4]
print(f"new solar built 2046-2050 within ${slack * 1e6:,.0f} of the least cost: {lo:.2f} to {hi:.2f} GW (the plan shown: {plan50:.2f} GW)")
assert lo - 0.01 <= plan50 <= hi + 0.01 and hi - lo > 0.5
print("answer: many plans cost almost the same; report what they share (cost, CO2, imports), and treat one plan's late builds as one of several")

In [ ]:
# worked answer 5: the step weights
STEP = run(dict(DEFAULT, acc="step")); show("central, step weights", STEP)
print("weights, linear:", sys_of(DEFAULT)["w"], "and 2025 x", sys_of(DEFAULT)["w0"], "· step:", sys_of(dict(DEFAULT, acc='step'))["w"], "and 2025 x", sys_of(dict(DEFAULT, acc='step'))["w0"])
assert STEP["co2"][0] >= CENTRAL["co2"][0] - 0.1
print("answer: step counts 2030's level for 2026-2030, so the falling years before it are under-counted; straight lines follow the path more closely")